# Alice: adaptación de nuestro SpeechGRU + CTC
Reutiliza el decodificador PyTorch del proyecto para estimar **secuencias de fonemas a partir de EEG de escucha**. El paper de Alice es la referencia del dataset; este notebook no replica su método. No usa `sound_power`, audio ni palabras como entradas del modelo.

Ejecuta en Colab, también desde VS Code conectado a Colab. Un participante, unos 91,5 MB de fuentes EEG/anotaciones más los recursos de g2p-en. Mismo `run.py`, modelo, lotes y métricas que en el repositorio. Los pesos se entrenan desde cero. Este perfil pequeño usa CPU con dos hilos y no necesita GPU.

La partición es por tramos del audiolibro: 190 secuencias train, 44 val y 36 test. Selección por CTC de validación; test solo al terminar. Es un experimento exploratorio con un participante, no una demostración de reconstrucción de texto o de habla intentada.

In [ ]:
#@title 1. Comprobar que estamos en Colab
from pathlib import Path
import os
import subprocess
import sys
try:
    import google.colab
except ImportError as error:
    raise RuntimeError('Conecta este notebook a Colab; no ejecutarlo en el kernel local del PC.') from error
COLAB_BASE = Path.cwd().resolve()
PROJECT = COLAB_BASE / 'tic_alice_colab'
print('Proyecto Alice:', PROJECT)
print('Este perfil usa CPU con dos hilos; no necesita GPU.')

In [ ]:
#@title 2. Copiar el código del proyecto (snapshot incluido)
import base64
import hashlib
import io
import json
from pathlib import PurePosixPath
from zipfile import ZipFile

SOURCE_SHA256 = '4c032b01cf6fd82fe9cb1bba5bdb7a98373729ed7c4dcaf9ce4c30a91c88843c'
SOURCE_B64 = (
    '...'
)

SOURCE_BYTES = base64.b64decode(SOURCE_B64)
if hashlib.sha256(SOURCE_BYTES).hexdigest() != SOURCE_SHA256:
    raise ValueError("El paquete de código no supera su SHA-256.")
PROJECT.mkdir(parents=True, exist_ok=True)
with ZipFile(io.BytesIO(SOURCE_BYTES)) as archive:
    for member in archive.infolist():
        relative = PurePosixPath(member.filename)
        if relative.is_absolute() or ".." in relative.parts or member.is_dir():
            raise ValueError(f"Ruta inesperada en el paquete: {member.filename}")
        destination = PROJECT.joinpath(*relative.parts)
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(archive.read(member))
SOURCE_MANIFEST = json.loads((PROJECT / "source_snapshot.json").read_text())
for name, expected in SOURCE_MANIFEST["files"].items():
    if hashlib.sha256((PROJECT / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Código distinto del snapshot: {name}")
print("Código preparado:", len(SOURCE_MANIFEST["files"]), "archivos")
print("Commit base:", SOURCE_MANIFEST["git_commit_base"])
print("SHA-256 del paquete:", SOURCE_SHA256)


In [ ]:
#@title 3. Reutilizar el entorno anterior o preparar uno aislado
import shutil
from collections import deque
from contextlib import nullcontext
import signal

# Solo se reutiliza el entorno de T12 si coinciden las dependencias; no se modifica.
candidate = COLAB_BASE / 'tic_proyecto_colab' / '.venv' / 'bin' / 'python'
probe = """
from importlib.metadata import version
expected = {'torch':'2.5.1','numpy':'2.1.3','scipy':'1.14.1','pyyaml':'6.0.2','wandb':'0.18.7','matplotlib':'3.9.3','g2p-en':'2.1.0','nltk':'3.9.2'}
assert all(version(k).split('+')[0] == v for k, v in expected.items())
"""
reusable = candidate.is_file() and subprocess.run([str(candidate), '-c', probe], capture_output=True).returncode == 0
if reusable:
    PYTHON = candidate
    print('Entorno anterior compatible; no se reinstala ni modifica.')
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', 'uv==0.9.4'], check=True)
    uv = shutil.which('uv')
    if uv is None:
        raise RuntimeError('No se encontró uv.')
    PYTHON = PROJECT / '.venv' / 'bin' / 'python'
    if not PYTHON.is_file():
        subprocess.run([uv, 'venv', '--python', '3.12.11', '--seed', str(PROJECT / '.venv')], check=True)
    subprocess.run([uv, 'pip', 'install', '--python', str(PYTHON), '--index-url', 'https://download.pytorch.org/whl/cpu', 'torch==2.5.1'], check=True)
    subprocess.run([uv, 'pip', 'install', '--python', str(PYTHON), '-r', 'requirements-alice.txt'], cwd=PROJECT, check=True)
    subprocess.run([uv, 'pip', 'check', '--python', str(PYTHON)], check=True)
    subprocess.run([str(PYTHON), '-c', probe], check=True)
for name in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS'):
    os.environ[name] = '2'

def ejecutar(argumentos: list[str], log: Path | None = None) -> None:
    """Muestra stdout/stderr, conserva log y detiene el proceso al interrumpir."""
    recent = deque(maxlen=25)
    process = subprocess.Popen([str(PYTHON), *argumentos], cwd=PROJECT, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1)
    try:
        with (log.open('w', encoding='utf-8') if log else nullcontext(None)) as stream:
            for line in process.stdout:
                print(line, end='', flush=True)
                recent.append(line)
                if stream:
                    stream.write(line)
                    stream.flush()
        returncode = process.wait()
    finally:
        if process.poll() is None:
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
        process.stdout.close()
    if returncode:
        raise RuntimeError('Falló el comando: ' + ' '.join(argumentos) + '\n' + ''.join(recent))
print('Entorno preparado. No hay que reiniciar el kernel.')

In [ ]:
#@title 4. Guardar este experimento en Drive
from datetime import datetime
from uuid import uuid4
from zoneinfo import ZoneInfo
from google.colab import drive
CARPETA_DRIVE = 'speech_bci' #@param {type:"string"}
relative = Path(CARPETA_DRIVE)
if not CARPETA_DRIVE.strip() or relative.is_absolute() or '..' in relative.parts:
    raise ValueError('Indica una carpeta relativa a Mi unidad.')
DRIVE_MOUNT = COLAB_BASE / 'drive'
drive.mount(str(DRIVE_MOUNT))
DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive' / relative
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
link = PROJECT / 'runs' / 'drive_alice'
link.parent.mkdir(parents=True, exist_ok=True)
if link.is_symlink():
    if link.resolve() != DRIVE_ROOT.resolve():
        raise RuntimeError('La carpeta de Drive difiere de la usada antes; utiliza una sesión nueva.')
elif link.exists():
    raise RuntimeError('Ya existe runs/drive_alice y no es el enlace esperado.')
else:
    link.symlink_to(DRIVE_ROOT, target_is_directory=True)
EXPERIMENT_ID = 'alice_' + datetime.now(ZoneInfo('Europe/Madrid')).strftime('%Y%m%d_%H%M%S') + '_' + uuid4().hex[:8]
RESULTS = DRIVE_ROOT / 'experimentos' / EXPERIMENT_ID
RESULTS.mkdir(parents=True, exist_ok=False)
CONFIG = 'configs/alice_ctc_ejecucion.yaml'
config_code = """
import sys
from pathlib import Path
import yaml
cfg = yaml.safe_load(Path('configs/alice_ctc.yaml').read_text())
base = Path('runs/drive_alice/experimentos') / sys.argv[1]
cfg['salida']['directorio'] = str(base / 'resultados')
cfg['wandb']['directorio'] = str(base / 'tracking')
Path(sys.argv[2]).write_text(yaml.safe_dump(cfg, sort_keys=False))
"""
subprocess.run([str(PYTHON), '-c', config_code, EXPERIMENT_ID, CONFIG], cwd=PROJECT, check=True)
(RESULTS / 'fuentes.zip').write_bytes(SOURCE_BYTES)
(RESULTS / 'source_snapshot.json').write_text(json.dumps(SOURCE_MANIFEST, indent=2))
shutil.copyfile(PROJECT / CONFIG, RESULTS / 'config.yaml')
print('Resultados y checkpoints:', RESULTS)
print('W&B: proyecto tiiac-2026, modo online.')

In [ ]:
#@title 5. Obtener EEG, preparar fonemas y fragmentos CTC
# No requiere los datos T12; solo reutiliza sus recursos públicos de pronunciación.
ejecutar(['scripts/get_alice.py', '--catalogo', 'docs/datasets/alice_eeg/catalogo_ctc.json'], RESULTS / 'descarga.log')
ejecutar(['scripts/get_phoneme_resources.py', '--config', CONFIG], RESULTS / 'recursos.log')
ejecutar(['scripts/prepare_alice_ctc.py', '--config', CONFIG], RESULTS / 'preparacion.log')

La celda siguiente reutiliza `WANDB_API_KEY` si sigue en la memoria del kernel. En otro caso pide una entrada oculta que también funciona en VS Code. Puedes activar Secretos si usas la web de Colab. No escribas la clave en el archivo ni en Git.

In [ ]:
#@title 6. Credencial de W&B
import getpass
USAR_SECRETOS_COLAB = False #@param {type:"boolean"}
api_key = os.environ.get('WANDB_API_KEY')
if not api_key and USAR_SECRETOS_COLAB:
    from google.colab import userdata
    api_key = userdata.get('WANDB_API_KEY')
if not api_key:
    api_key = getpass.getpass('API key de W&B (entrada oculta): ')
if not api_key or not api_key.strip():
    raise ValueError('Falta WANDB_API_KEY.')
os.environ['WANDB_API_KEY'] = api_key.strip()
os.environ['WANDB_MODE'] = 'online'
del api_key
print('Credencial preparada; la conexión se realiza al abrir el run.')

In [ ]:
#@title 7. Entrenar, evaluar y verificar la recarga sin entrenar otra vez
import math
if not os.environ.get('WANDB_API_KEY'):
    raise RuntimeError('Ejecuta la celda de credencial.')
before = set((RESULTS / 'resultados').glob('*/resultado.json'))
ejecutar(['run.py', '--config', CONFIG], RESULTS / 'entrenamiento.log')
after = set((RESULTS / 'resultados').glob('*/resultado.json')) - before
if len(after) != 1:
    raise RuntimeError('No se identifica un único resultado nuevo. Revisa el log guardado.')
RUN_DIR = next(iter(after)).parent
report = json.loads((RUN_DIR / 'resultado.json').read_text())
checkpoint = PROJECT / report['checkpoint']
EVAL_CONFIG = 'configs/alice_ctc_reevaluacion.yaml'
code = """
from pathlib import Path
import sys, yaml
cfg = yaml.safe_load(Path(sys.argv[1]).read_text())
cfg['experimento']['nombre'] += '_recarga'
Path(sys.argv[2]).write_text(yaml.safe_dump(cfg, sort_keys=False))
"""
subprocess.run([str(PYTHON), '-c', code, CONFIG, EVAL_CONFIG], cwd=PROJECT, check=True)
before_eval = set((RESULTS / 'resultados').glob('*/resultado.json'))
ejecutar(['run.py', '--config', EVAL_CONFIG, '--solo-evaluar', '--checkpoint', str(checkpoint)], RESULTS / 'recarga.log')
evaluations = set((RESULTS / 'resultados').glob('*/resultado.json')) - before_eval
if len(evaluations) != 1:
    raise RuntimeError('No se identifica la reevaluación.')
reevaluated = json.loads(next(iter(evaluations)).read_text())
assert reevaluated['evaluation_only'] and reevaluated['completed_steps'] == 0
assert report['test']['per'] == reevaluated['test']['per']
assert math.isclose(report['test']['ctc_loss'], reevaluated['test']['ctc_loss'], rel_tol=1e-6, abs_tol=1e-5)
ejecutar(['scripts/report_alice_ctc.py', '--run-dir', str(RUN_DIR)], RESULTS / 'figura.log')
print('Entrenamiento y recarga comprobados:', RUN_DIR)

In [ ]:
#@title 8. Mostrar resultados guardados (no entrena)
from IPython.display import Image, display
report = json.loads((RUN_DIR / 'resultado.json').read_text())
print('Pasos completados:', report['completed_steps'], '| mejor val:', report['best_val_step'])
print('Bucle con validación y guardados (s):', report['training_seconds_including_val_and_saves'])
print('PER en test:', f"{report['test']['per']:.2%}")
print('PER sin señal EEG:', f"{report['zero_eeg_control']['per']:.2%}")
print('PER del prior de fonemas de train:', f"{report['phone_prior_control']['per']:.2%}")
print('Referencias de test:', report['test']['frases'], '| fonemas:', report['test']['fonemas'])
display(Image(filename=str(RUN_DIR / 'comparacion_ctc.png')))

El límite suave del bucle es 180 segundos para este perfil; instalación, preparación, evaluación final, guardado y sincronización añaden tiempo. Se registran los tiempos y pasos realmente completados. Los pesos elegidos por validación se guardan en `mejor_val.pt` y se comprueban con `--solo-evaluar` en un segundo run, sin actualizaciones.

Las etiquetas de fonemas proceden de la transcripción con el mismo g2p-en de T12. CTC aprende una alineación monótona; no recibe tiempos exactos de fonemas. Los tiempos de palabras publicados delimitan los fragmentos EEG y sus referencias. No hay modelo de lenguaje ni cálculo de WER. El PER cuenta SIL y puede superar el 100 %.

Fuentes: [dataset NEMAR nm000180](https://www.nemar.org/dataset/nm000180), [artículo del dataset](https://doi.org/10.1371/journal.pone.0207741). El método adaptado y los cambios respecto a T12 se documentan en `docs/experimentos/alice_ctc.md` del repositorio. La interpretación y discusión de la entrega corresponden al autor.